In [ ]:
# 必要なライブラリをインストール
!pip install genesis-world transformers torch pillow opencv-python imageio

import genesis as gs
import numpy as np
import imageio
import cv2
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel
from IPython.display import Video

# 1. AIモデル（CLIP）を読み込む
print("AIモデルを読み込み中")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
scene = gs.Scene(show_viewer=False)

# 最初に地面（Plane）を配置する
ground_plane = scene.add_entity(gs.morphs.Plane())

# 関節（joint）が入っているFrankaのMJCF（XML）ファイルを読み込んで使うことにした
franka = scene.add_entity(gs.morphs.MJCF(file='xml/franka_emika_panda/panda.xml'))

# 3つの物体を全部同じオレンジ色にして配置する
orange_color = gs.surfaces.Default(color=(0.95, 0.55, 0.15, 1.0))

# ① ボール: 右
ball = scene.add_entity(
    gs.morphs.Sphere(radius=0.055, pos=(0.45, -0.25, 0.055), fixed=True),
    surface=orange_color
)

# ② 箱: 中
box = scene.add_entity(
    gs.morphs.Box(size=(0.09, 0.09, 0.09), pos=(0.50, 0.00, 0.045), fixed=True),
    surface=orange_color
)

# ③ 円柱: 左
cylinder = scene.add_entity(
    gs.morphs.Cylinder(radius=0.045, height=0.14, pos=(0.45, 0.25, 0.07), fixed=True),
    surface=orange_color
)

# カメラをセットする
cam = scene.add_camera(
    res=(640, 480),
    pos=(1.45, 0.0, 0.85),
    lookat=(0.35, 0.0, 0.15),
    fov=45
)

scene.build()

# 3. AI（CLIP）の認識結果から目標の向き（angular_zに相当する角度）を決める！
prompt_list = ["a round sphere ball", "a square cube block", "a tall cylinder pillar", "a round sphere ball (2nd)"]
name_list = ["SPHERE (Ball)", "CUBE (Box)", "CYLINDER (Pillar)", "SPHERE (Ball)"]

# 記事の Left(0.5), Up(0.0), Right(-0.5) の angular.z の考え方を参考に、関節の回転角を設定
angular_z_list = [-0.48, 0.0, 0.48, -0.48]

# 最初のポーズ（base_linkから各jointの初期角度）
home_qpos = np.array([0.0, -0.4, 0.0, -1.8, 0.0, 1.5, 0.7, 0.04, 0.04])
franka.set_qpos(home_qpos)
scene.step()

# 動画のフレームをためるリスト
frames = []
print("開始")

# 4. 4つのミッションを順番に実行する
for i in range(4):
    target_prompt = prompt_list[i]
    target_name = name_list[i]

    # 記事の /cmd_vel のパラメータ名（linear_x, angular_z）を真似して変数を作る
    cmd_vel_angular_z = angular_z_list[i]
    cmd_vel_linear_x = 0.5  # 前進する動きのイメージ

    # カメラで今の画像を撮る（タプルの0番目を取り出す）
    render_out = cam.render()
    rgb_now = render_out[0]

    # 画像から3つの物体の場所を切り抜く
    crop1 = Image.fromarray(rgb_now[230:408, 352:544]) # 右（ボール）
    crop2 = Image.fromarray(rgb_now[230:408, 256:384]) # 中（箱）
    crop3 = Image.fromarray(rgb_now[201:408, 96:288])  # 左（円柱）
    crops = [crop1, crop2, crop3]

    # CLIPでどの画像がプロンプトに近いか計算する
    texts = [target_prompt, "empty flat grey floor background"]
    inputs = clip_processor(text=texts, images=crops, return_tensors="pt", padding=True)
    outputs = clip_model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1)[:, 0].detach().numpy()

    # 一番スコアが高いものを選ぶ
    best_idx = np.argmax(probs)
    conf = float(probs[best_idx])

    # 目標のポーズを作る
    reach_qpos = np.array([cmd_vel_angular_z, cmd_vel_linear_x + 0.05, 0.0, -1.35, 0.0, 1.9, 0.7, 0.0, 0.0])

    # 525ステップ動かして録画する
    for step in range(525):
        if step < 90:
            # 待ち
            qpos = home_qpos
            status = "Waiting"
        elif step < 240:
            # ターゲットに向かって少しずつ動かす
            wariai = (step - 90) / 150.0
            qpos = (1 - wariai) * home_qpos + wariai * reach_qpos
            status = "Phase 2: pub /cmd_vel -> Approaching " + target_name
        elif step < 375:
            # 先端のjoint（revolute）を動かす
            qpos = reach_qpos.copy()
            qpos[6] = qpos[6] + 0.4 * np.sin((step - 240) * 0.15)
            status = "Phase 3: Inspecting " + target_name
        else:
            # 元の場所に戻る
            wariai = (step - 375) / 150.0
            qpos = (1 - wariai) * reach_qpos + wariai * home_qpos
            status = "Phase 4: Returning to base_link Home"

        franka.set_qpos(qpos)
        scene.step()

        # カメラで撮影
        render_out = cam.render()
        rgb = render_out[0]

        # OpenCVで文字を入れるためにRGBからBGRに変える
        img_bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

        # 上の方に黒い四角を描いて文字を入れる
        cv2.rectangle(img_bgr, (10, 10), (630, 115), (20, 20, 20), -1)

        text1 = "Mission " + str(i+1) + "/4 | Prompt: " + target_prompt
        text2 = "CLIP: " + target_name + " (Score: " + str(round(conf, 2)) + ") | angular.z: " + str(cmd_vel_angular_z)

        cv2.putText(img_bgr, text1, (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        cv2.putText(img_bgr, text2, (20, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 255, 100), 1)
        cv2.putText(img_bgr, status, (20, 95), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 220, 255), 1)
        cv2.putText(img_bgr, "2x Speed", (530, 460), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 2)

        # 色をRGBに戻してリストに追加する
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        frames.append(img_rgb)

# 5. Macでも再生できるようにmp4で保存する
imageio.mimsave('final_shape_robot.mp4', frames, fps=30, codec='libx264', pixelformat='yuv420p')
print("動画保存完了")

# 動画を表示して確認する
Video('final_shape_robot.mp4', embed=True, width=600)